# Model Tester

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Setup

In [2]:
import os
import sys
import time

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.losses import MeanSquaredError, mse
from tensorflow.keras.callbacks import (
    EarlyStopping,
    TerminateOnNaN,
    TensorBoard,
    ModelCheckpoint,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay, LearningRateSchedule

sys.path.append(os.path.join(os.getcwd(), ".."))
from scripts import Core
from scripts.models import get_model_class
from scripts.utils import setup_logging, load_data, get_fisher, try_init_wandb
from scripts.utils.plots import plot_predictions, plot_histogram
from scripts.utils.tf.dataloaders import *
from scripts.utils.tf.plots import plot_metrics
from scripts.utils.tf.callbacks import TimedLoggingCallback, WarmupLearningRate

2024-05-01 14:17:33.569772: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-01 14:17:33.569827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-01 14:17:33.570920: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-01 14:17:33.578586: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-01 14:17:34.887488: W tensorflow/compiler/tf2

01-May-24 14:17:38 - scripts.models.register - DEBUG - Registering Model: ALM
01-May-24 14:17:38 - scripts.models.register - DEBUG - Registering Model: DCNN
01-May-24 14:17:38 - scripts.models.register - DEBUG - Registering Model: ISENSEE
01-May-24 14:17:38 - scripts.models.register - DEBUG - Registering Model: RESNET
01-May-24 14:17:38 - scripts.models.register - DEBUG - Registering Model: NAGARAJAPPA


In [3]:
logger = setup_logging(__name__, level=logging.INFO)
logging.getLogger("scripts").setLevel(logging.DEBUG)
logging.getLogger("tensorflow").setLevel(logging.ERROR)

## Configure

In [4]:
# core = Core(["settings/planck.json", "--nsims", "100"], trainer=True)
core = Core(["settings/heidelberg.json", "--nsims", "100"], trainer=True)

MAX_EPOCHS = 20
BATCH_SIZE = 32

# just some info for the model name
timestamp = int(time.time())
model_settings = {
    "name": f"tester_{core.base_name}_{timestamp}",
}

data_settings = {
    "shuffle": False,
    "shuffle_buffer": 1000,
    "seed": None,
    "batch_size": BATCH_SIZE,
    "cache": True,
    "normalize": False,
}

# additional metrics we are interested in
metrics = ["mean_absolute_error"]

callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    TimedLoggingCallback(print_frequency=3),
    # TensorBoard(log_dir=f"{s.tb_dir}/{model_settings['name']}"),
    TerminateOnNaN(),
]

01-May-24 14:17:38 - scripts.core - DEBUG - Parsing CLI args: ['settings/heidelberg.json', '--nsims', '100']
01-May-24 14:17:38 - scripts.core - INFO - Loading settings from file settings/heidelberg.json
01-May-24 14:17:38 - scripts.core - DEBUG - Forcing setting 'nsims' to '100' due to CLI
01-May-24 14:17:38 - scripts.core - DEBUG - Found non-default value for 'cosmo_params': {'H0': 70.1, 'As': 2.457e-09, 'ns': 0.96, 'ombh2': 0.02256, 'omch2': 0.1143, 'tau': 0.084, 'max_l': 1500, 'lmax': 1024} (default: {'As': 2.13e-09, 'ns': 0.9624, 'pivot_scalar': 0.05, 'max_l': 1000, 'lmax': 500})
01-May-24 14:17:38 - scripts.core - DEBUG - Overriding cosmo param As from 2.13e-09 to 2.457e-09
01-May-24 14:17:38 - scripts.core - DEBUG - Overriding cosmo param ns from 0.9624 to 0.96
01-May-24 14:17:38 - scripts.core - DEBUG - Overriding cosmo param max_l from 1000 to 1500
01-May-24 14:17:38 - scripts.core - DEBUG - Overriding cosmo param lmax from 500 to 1024
01-May-24 14:17:38 - scripts.core - INFO 

In [5]:
cb = try_init_wandb()
if cb is not None:
    callbacks.append(cb)

01-May-24 14:17:39 - git.cmd - DEBUG - Popen(['git', 'version'], cwd=/users/stevensonb/Research/MLPNG, stdin=None, shell=False, universal_newlines=False)
01-May-24 14:17:39 - git.cmd - DEBUG - Popen(['git', 'version'], cwd=/users/stevensonb/Research/MLPNG, stdin=None, shell=False, universal_newlines=False)
01-May-24 14:17:39 - wandb.docker.auth - DEBUG - Trying paths: ['/users/stevensonb/.docker/config.json', '/users/stevensonb/.dockercfg']
01-May-24 14:17:39 - wandb.docker.auth - DEBUG - No config file found
01-May-24 14:17:39 - sentry_sdk.errors - DEBUG - [Tracing] Create new propagation context: {'trace_id': '001a0e83ce9e40888b708ac446370e3e', 'span_id': '81f6aa6684613df0', 'parent_span_id': None, 'dynamic_sampling_context': None}
01-May-24 14:17:52 - urllib3.connectionpool - DEBUG - Starting new HTTP connection (1): bcm-dgxa100-0005:8888
01-May-24 14:17:52 - urllib3.connectionpool - DEBUG - http://bcm-dgxa100-0005:8888 "GET /api/sessions?token=4f0f639990566c098e4fb92732eb8b33d06906

wandb: Currently logged in as: jbrandons (mlpng). Use `wandb login --relogin` to force relogin


01-May-24 14:17:54 - git.cmd - DEBUG - Popen(['git', 'cat-file', '--batch-check'], cwd=/users/stevensonb/Research/MLPNG, stdin=<valid stream>, shell=False, universal_newlines=False)


## Isensee Model

In [6]:
model = get_model_class("ISENSEE")(core)
train_ds, test_ds, val_ds = model.dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model.make_model(**model_settings)
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

01-May-24 14:17:59 - scripts.utils.tf.dataloaders - DEBUG - auto detected num_replicas: 1
01-May-24 14:17:59 - scripts.utils.tf.dataloaders - INFO - Loading data/patches/heidelbergx10000x2.hdf5 with 20000 samples, and data shape (512, 512, 1)
01-May-24 14:18:01 - scripts.utils.tf.dataloaders - DEBUG - Dataset sizes: Training 16000, Testing 2000, Validation 2000


2024-05-01 14:18:01.232214: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
2024-05-01 14:18:01.235203: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 78911 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:07:00.0, compute capability: 8.0


Model: "tester_heidelbergx10000_1714591058"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 512, 512, 1)]        0         []                            
                                                                                                  
 periodic_padding2d (Period  (None, 514, 514, 1)          0         ['input_1[0][0]']             
 icPadding2D)                                                                                     
                                                                                                  
 conv2d (Conv2D)             (None, 512, 512, 16)         160       ['periodic_padding2d[0][0]']  
                                                                                                  
 group_normalization (Group  (None, 512, 512, 16)         32     

2024-05-01 14:18:03.692799: W tensorflow/core/grappler/optimizers/data/auto_shard.cc:553] The `assert_cardinality` transformation is currently not handled by the auto-shard rewrite and will be removed.
2024-05-01 14:19:21.356073: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2024-05-01 14:19:26.810905: I external/local_xla/xla/service/service.cc:168] XLA service 0x155101b24000 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-05-01 14:19:26.810935: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2024-05-01 14:19:26.816475: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1714591166.914551 1598396 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the proce

499/500 [=============================>] - ETA:  1s - loss: 328988.7500 - mean_absolute_error: 496.7589047

2024-05-01 14:24:39.017704: W tensorflow/core/grappler/optimizers/data/auto_shard.cc:553] The `assert_cardinality` transformation is currently not handled by the auto-shard rewrite and will be removed.


Epoch: 1/20 - Time: 7:44 - loss: 329089.0000 - mean_absolute_error: 496.8893 - val_loss: 341253.4688 - val_mean_absolute_error: 510.6555
Epoch: 2/20 - Time: 5:04 - loss: 328032.5625 - mean_absolute_error: 495.8374 - val_loss: 341410.3438 - val_mean_absolute_error: 510.4641
Epoch: 3/20 - Time: 5:04 - loss: 324037.2188 - mean_absolute_error: 492.2125 - val_loss: 342853.6562 - val_mean_absolute_error: 510.3497
Epoch: 4/20 - Time: 5:04 - loss: 318531.2500 - mean_absolute_error: 486.8480 - val_loss: 346354.9062 - val_mean_absolute_error: 512.3768
180/500 [==========>...................] - ETA: 3:09 - loss: 327492.7188 - mean_absolute_error: 495.5609

KeyboardInterrupt: 

In [ ]:
fisher = get_fisher(core.alm_file)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## ALM Model

In [ ]:
model = get_model_class("ALM")(core)
train_ds, test_ds, val_ds = model.dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model.make_model(**model_settings)
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

In [ ]:
fisher = get_fisher(core.alm_file)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## DCNN Model

In [ ]:
model = get_model_class("dcnn")(core)
train_ds, test_ds, val_ds = model.dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model.make_model(**model_settings)
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)


In [ ]:
fisher = get_fisher(core.alm_file)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## RESNET

In [ ]:
model = get_model_class("RESNET")(core)
train_ds, test_ds, val_ds = model.dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model.make_model(**model_settings)
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

In [ ]:
fisher = get_fisher(core.alm_file)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## NAGARAJAPPA

In [ ]:
model = get_model_class("NAGARAJAPPA")(core)
train_ds, test_ds, val_ds = model.dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    opt = Adam(learning_rate)
    model.make_model(**model_settings)
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

In [ ]:
fisher = get_fisher(core.alm_file)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])
plot_metrics(history, metrics=["loss"] + metrics)
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)